In [ ]:
import io
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

import seaborn as sns
from PIL import Image
from datasets import load_dataset
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"
DATASET_NAME = "clip-benchmark/wds_imagenetv2"
SPLIT = "test"
SAMPLE_SIZE = 10
THUMB_SIZE = (80, 80)
OUT_PATH = "heatmap.png"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()

In [ ]:
print(model)

In [ ]:
# 데이터셋을 불러오고, test split에서 섞어서 10개만 고름
dataset = load_dataset(DATASET_NAME)
ds = dataset[SPLIT]
subset = ds.shuffle(seed=2026).select(range(SAMPLE_SIZE))

In [ ]:
cls2label = [line.strip() for line in open("./dataset/classnames.txt", "r", encoding="utf-8").readlines()]
cls2label

# ['tench',
#  'goldfish',
#  'great white shark',
#  'tiger shark',
#  ...
# ]

In [ ]:
def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")
        # subset에서 webp 컬럼을 꺼내 PIL 이미지 리스트를 만듦
        # images = [to_pil(x) for x in list(subset("webp"))]
    if isinstance(x, dict) and "bytes" in x and x["bytes"] is not None:
        return Image.open(io.BytesIO(x["bytes"])).convert("RGB")
    if isinstance(x, str):
        return Image.open(x).convert("RGB")
    raise TypeError(f'Unsupported image type: {type(x)}')

In [ ]:
label_names = [cls2label[int(c)] for c in list(subset['cls'])]
label_texts = [f'a photo of a {name}' for name in label_names]

images = [to_pil(x) for x in list(subset['webp'])]

In [ ]:
images

In [ ]:
label_texts

In [ ]:
with torch.no_grad():
    # 이미지 임베딩
    # processor가 이미지를 CLIP이 받는 파이토치 텐서 형태로 변환 (batch, 3, H, W)
    inputs_image = processor(images=images, return_tensors='pt', padding=True).to(device)
    # vision_model이 비전 트랜스포머를 통과시켜 출력
    vision_out = model.vision_model(**inputs_image)
    # pooler_output = "대표 임베딩" 같은 역할을 하는 벡터(보통 CLS 토큰 기반)
    # visual_projection으로 CLIP 공통 임베딩 공간에 차원에 맞춰 projection 함
    image_features = model.visual_projection(vision_out.pooler_output)

    # 텍스트 임베딩
    # 텍스트도 토크나이징/패딩 후 모델 통과
    inputs_text = processor(text=label_texts, return_tensors='pt', padding=True).to(device)
    text_out = model.text_model(**inputs_text)
    text_features = model.text_projection(text_out.pooler_output)

# 정규화 + 유사도 계산(코사인 유사도)
# 각 벡터를 L2 정규화하면 길이가 1이 됨
# 내적(@)이 코사인 유사도
# 결과: (10, D) @ (D, 10) = (10, 10)
image_features = image_features / image_features.norm(dim=-1, keepdim=True)
text_features = text_features / text_features.norm(dim=-1, keepdim=True)
similarity_matrix = (image_features @ text_features.T).cpu().numpy()

In [ ]:
def create_thumbnail(img, size=(80, 80)):
    return img.resize(size)

In [ ]:
thumbnails = [create_thumbnail(img, THUMB_SIZE) for img in images]
thumbnails

In [ ]:
fig, ax = plt.subplots(figsize=(20, 12))
im = ax.imshow(similarity_matrix, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)

ax.set_xticks(np.arange(len(label_texts)))
ax.set_xticklabels(label_texts, rotation=45, ha='right')

ax.set_yticks(np.arange(len(images)))
ax.set_yticklabels([""] * len(images))

ax.set_title('CLIP Image vs. Label Similarity (Cosine Similarity)')
ax.set_xlabel('Label Text')
ax.set_ylabel('Image')

for i in range(similarity_matrix.shape[0]):
    for j in range(similarity_matrix.shape[1]):
        ax.text(j, i, f'{similarity_matrix[i, j]:.2f}', ha='center', va='center', fontsize=8)

#
for i, img in enumerate(thumbnails):
    imagebox = OffsetImage(img, zoom=1.0)
    # AnnotationBbox를 사용해서 좌표 (-0.6, i) 위치에 썸네일 이미지를 붙임
    # (-0.6, i)는 히트맵의 첫 열(0)보다 왼쪽에 배치하기 위해서
    ab = AnnotationBbox(imagebox, (-0.6, i), frameon=False, xycoords='data', boxcoords='data', pad=0)
    ax.add_artist(ab)

ax.set_xlim(-1.2, similarity_matrix.shape[1] - 0.5)
ax.set_ylim(similarity_matrix.shape[0] - 0.5, -0.5)

plt.tight_layout()
plt.savefig(OUT_PATH, dpi=200)
plt.show()
print(f'saved: {OUT_PATH}')